# Week 3 Short-Term Memory Experiment

Compare separate stateless invocations with thread-scoped conversation memory.

In [1]:
import sys
from pathlib import Path
from pprint import pprint

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.messages import AIMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI

project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / '.venv' / 'Scripts' / 'python.exe').exists():
    project_root = project_root.parent

load_dotenv(project_root / '.env')
expected_python = (project_root / '.venv' / 'Scripts' / 'python.exe').resolve()
assert Path(sys.executable).resolve() == expected_python
assert sys.version_info[:3] == (3, 13, 1)

model = ChatGoogleGenerativeAI(model='gemini-3.1-flash-lite')
system_instruction = (
    'Answer using only information available in the current conversation. '
    'If the user has not provided the requested information, say that you do not know instead of guessing.'
)
first_message = 'My name is Abdelmalek, and I am preparing a conference presentation about vehicle tyre-pressure monitoring using machine learning.'
follow_up_question = 'What is my name, and what is my conference presentation about?'
print('Using the project .venv with Python 3.13.1 and Gemini Flash-Lite.')

Using the project .venv with Python 3.13.1 and Gemini Flash-Lite.


## Experiment 1: Without a checkpointer

In [2]:
no_memory_agent = create_agent(model=model, system_prompt=system_instruction)
no_memory_first = no_memory_agent.invoke({'messages': [HumanMessage(content=first_message)]})
first_no_memory_answer = no_memory_first['messages'][-1]
print('First invocation response:')
pprint(first_no_memory_answer)

First invocation response:
AIMessage(content=[{'type': 'text', 'text': "Hello, Abdelmalek. That sounds like a fascinating topic for a conference presentation. Since you haven't provided any specific details about your research or the data you are using yet, I do not know the specific methodologies or findings you plan to discuss.\n\nHow can I help you with your presentation? For example, are you looking for help with the structure, explaining specific machine learning concepts, or drafting sections of your talk?", 'extras': {'signature': 'EnMKcQFpFH0TbBcfkvZ/m+SRb7NuirzwPnDW20veo/5MXfXS8FRO9YTDUj8pdqPLfXUU0A4Wo8OrGJDcRgjo+dqcJGi+fhFIgPbxvQgJSB1shetWx5hE9SokaVYu/32mPW96uzY4XGDQmaQKLoo0ByDYgC0Y'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0fd9d-8aea-72d0-8dd5-fcd9c0194821-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 55, 'out

In [3]:
no_memory_second_input = {'messages': [HumanMessage(content=follow_up_question)]}
no_memory_second = no_memory_agent.invoke(no_memory_second_input)
second_no_memory_answer = no_memory_second['messages'][-1]
print('Second invocation response:')
pprint(second_no_memory_answer)
print('Messages explicitly passed to the second invocation:', len(no_memory_second_input['messages']))
first_turn_present_in_second_input = any(
    isinstance(message, HumanMessage) and message.content == first_message
    for message in no_memory_second_input['messages']
)
print('First invocation history passed to the second invocation:', first_turn_present_in_second_input)

Second invocation response:
AIMessage(content=[{'type': 'text', 'text': 'I do not know your name or the topic of your conference presentation, as you have not provided that information in this conversation.', 'extras': {'signature': 'EnMKcQFpFH0T7pidIY5piAjc7urVv3D/tJKaygyXrLMz++yinQVXWwU2WjiwjrzTo5F/tSvmwEJbKv7Vca+0NvkKuHx7fQVp8mrAohQ2I8lPVIpV1Z94xvsozz4tFCAptrvfdRJBOv/1zDa4HCAG7XbF6l9H'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0fd9d-a2ea-7c33-8674-43b027e70c6a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 44, 'output_tokens': 25, 'total_tokens': 69, 'input_token_details': {'cache_read': 0}})
Messages explicitly passed to the second invocation: 1
First invocation history passed to the second invocation: False


## Experiment 2: Short-term memory on the same thread

In [4]:
from langgraph.checkpoint.memory import InMemorySaver

memory_checkpointer = InMemorySaver()
memory_agent = create_agent(
    model=model,
    system_prompt=system_instruction,
    checkpointer=memory_checkpointer,
)
config = {'configurable': {'thread_id': 'abdelmalek-session-1'}}
memory_first = memory_agent.invoke(
    {'messages': [HumanMessage(content=first_message)]},
    config=config,
)
print('First invocation response with memory enabled:')
pprint(memory_first['messages'][-1])

First invocation response with memory enabled:
AIMessage(content=[{'type': 'text', 'text': "Hello, Abdelmalek. That sounds like a very interesting topic for a conference presentation. Since you haven't provided any specific details about your work yet, I do not know the specifics of your machine learning approach or the particular focus of your presentation. \n\nIf you would like to share more details, I would be happy to help you refine your content, structure your slides, or practice your explanation.", 'extras': {'signature': 'EnMKcQFpFH0T42wDGiIemF0Ki16sswaUdIASx5lwbY70e+RqutfGmzUMyJAnejU+DsH5OfKUPKmhizfH58NjtJ6hUBol1Piz05OoeaCF4w9H03v/LOKR6jmTSHkyt718tbeKRWR517LcfkHZDnpMSUmK5MeR'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0fd9d-ba89-7e03-8464-d7340d010221-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 55, 'output_token

In [5]:
memory_second = memory_agent.invoke(
    {'messages': [HumanMessage(content=follow_up_question)]},
    config=config,
)
memory_second_messages = memory_second['messages']
memory_answer = memory_second_messages[-1]
print('Final response with the same thread_id:')
pprint(memory_answer)

labels = {
    0: 'EARLIER HUMAN MESSAGE',
    1: 'EARLIER AI RESPONSE',
    len(memory_second_messages) - 2: 'NEW HUMAN QUESTION',
    len(memory_second_messages) - 1: 'FINAL AI ANSWER',
}
print('Every message returned in the second invocation:')
for index, message in enumerate(memory_second_messages):
    print(f'\n--- {labels.get(index, type(message).__name__.upper())} ({type(message).__name__}) ---')
    pprint(message)

answer_text = memory_answer.content if isinstance(memory_answer.content, str) else str(memory_answer.content)
remembers_name = 'abdelmalek' in answer_text.lower()
topic_text = answer_text.lower()
remembers_topic = all(term in topic_text for term in ('tyre', 'pressure', 'monitor')) and ('machine learning' in topic_text or 'machine-learning' in topic_text)
print('Correctly remembers the name:', remembers_name)
print('Correctly remembers the presentation topic:', remembers_topic)
print('Stored message count:', len(memory_second_messages))
print('Stored message types:', [type(message).__name__ for message in memory_second_messages])

Final response with the same thread_id:
AIMessage(content=[{'type': 'text', 'text': 'Your name is Abdelmalek, and your conference presentation is about vehicle tyre-pressure monitoring using machine learning.', 'extras': {'signature': 'EnMKcQFpFH0TZ3GCDyxPUOdTsdlPUXQrpwR5hmMlsDnfAsJQ0IVW5TIIGoC9eSBkRuMoijRAbflEuVTAuJqo40dyM6abbAwbb5NdsvBDWYNN29DVBSEwc3F0seWPdvWLYqZVfOWE86b1L9onKlWH2Fv2MgAW'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0fd9d-e5aa-7043-a13b-fbfaa4ffbbbf-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 152, 'output_tokens': 22, 'total_tokens': 174, 'input_token_details': {'cache_read': 0}})
Every message returned in the second invocation:

--- EARLIER HUMAN MESSAGE (HumanMessage) ---
HumanMessage(content='My name is Abdelmalek, and I am preparing a conference presentation about vehicle tyre-pressure monitoring usi

## Experiment 3: Change the thread

In [6]:
new_thread_config = {'configurable': {'thread_id': 'abdelmalek-session-2'}}
new_thread_result = memory_agent.invoke(
    {'messages': [HumanMessage(content=follow_up_question)]},
    config=new_thread_config,
)
new_thread_answer = new_thread_result['messages'][-1]
print('Response in abdelmalek-session-2:')
pprint(new_thread_answer)
prior_turn_in_new_thread = any(
    isinstance(message, HumanMessage) and message.content == first_message
    for message in new_thread_result['messages']
)
print('First-thread message available in the new thread:', prior_turn_in_new_thread)
print('New-thread message types:', [type(message).__name__ for message in new_thread_result['messages']])

Response in abdelmalek-session-2:
AIMessage(content=[{'type': 'text', 'text': 'I do not know your name, nor do I know what your conference presentation is about, as you have not provided that information in this conversation.', 'extras': {'signature': 'EnMKcQFpFH0T4cK/obO3xLM8szhdoBf+uvJN7UE2Ut4e1Cb2htfdof+nEIPgk5EuDUvWGer2Hb7BJgNPStyIevJtuAunDRzD7x6QloHki0SqQtve24wJuTegUiEuXMsw4eZCo6iNZZFaV8HcFuu70F6OTIN1'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0fd9d-f85e-79d3-a153-9101e6530e82-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 44, 'output_tokens': 29, 'total_tokens': 73, 'input_token_details': {'cache_read': 0}})
First-thread message available in the new thread: False
New-thread message types: ['HumanMessage', 'AIMessage']


## Observations

- **Behaviour without a checkpointer**  
  The second invocation received only the new question, so the agent did not know my name or presentation topic. This showed that separate model calls do not automatically share conversation history.

- **Behaviour with the same thread_id**  
  After adding InMemorySaver and using the same thread_id for both invocations, the agent correctly remembered my name and that my conference presentation concerns vehicle tyre-pressure monitoring using machine learning.

- **What appeared in the stored message list**  
  The returned state contained four messages in order: the first human message, the first AI response, the second human question and the final AI answer. The earlier conversation was therefore included when the second request was processed.

- **Behaviour after changing thread_id**  
  Using a different thread_id created a separate conversation. The agent could not access information stored under the original thread and correctly said that it did not know the details.

- **Limitation of InMemorySaver**  
  InMemorySaver stores conversation state only in the current process memory. The information disappears when the notebook kernel or application process is restarted, so a persistent database or checkpointer would be required for a production application.